<a href="https://colab.research.google.com/github/nasif-raihan/GSU/blob/main/CSCI_7090_Module10_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10 — Homework
**Computer Vision · SSD (Single Shot Detector)**

Your feature-map sizes, boxes-per-cell, class count, scale, ratio, offsets, and confidence scores are generated from your **roll number**, so your numbers are unique — pasted generic answers will not match your outputs.

## Rules
- Set `ROLL_NUMBER` at the top, then run the setup cell. Do **not** edit the parameter lines.
- Each question is a **task**: write the code yourself and store the result in the named variable(s) under **Produce**. There are no fill-in-the-blank stubs.
- Answer the **Written** prompts (Q7) in your own words. Run the final **Submission Summary** and keep its output.

## Grading (100): Q1 15 · Q2 15 · Q3 10 · Q4 20 · Q5 10 · Q6 15 · Q7 15  ·  ~1.5 hours

## Setup — run first
Generates your SSD parameters (feature-map sizes, boxes-per-cell, class count, scale/ratio, offsets) and a per-box confidence array for Q6.

In [1]:
# === SETUP — run first ===
ROLL_NUMBER = "901503252"
import hashlib, numpy as np
seed = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**32)
rng  = np.random.default_rng(seed)

# ---- Your personal parameters (unique to your roll number) — DO NOT EDIT ----
FMAPS = sorted(rng.choice([38,19,10,5,3,1], size=int(rng.choice([3,4])), replace=False).tolist(), reverse=True)
BPC   = [int(rng.choice([4,6])) for _ in FMAPS]          # default boxes per cell, per feature map
NUM_CLASSES = int(rng.choice([11, 21]))                   # 10/20 classes + background
SCALE = round(float(rng.uniform(0.2, 0.9)), 2)            # default-box scale (fraction of image)
RATIO = float(rng.choice([0.5, 1.0, 2.0, 3.0]))          # aspect ratio
DEFAULT_BOX = np.array([0.5, 0.5, 0.3, 0.2])              # (cx,cy,w,h), normalized
OFFSETS     = np.round(rng.uniform(-0.2, 0.2, size=4), 3) # predicted (dcx,dcy,dw,dh)
CONF_THRESH = round(float(rng.choice([0.3, 0.5, 0.7])), 2)  # confidence threshold for Q6

# Per-default-box top-class confidence scores for Q6 (one score per default box)
rng9   = np.random.default_rng(seed + 9)
_total = sum(f*f*b for f, b in zip(FMAPS, BPC))
CONF   = np.round(rng9.uniform(0.0, 1.0, _total), 3)

print("Roll:", ROLL_NUMBER, "| seed:", seed)
print("FMAPS:", FMAPS, "| BPC:", BPC, "| NUM_CLASSES:", NUM_CLASSES)
print("SCALE:", SCALE, "| RATIO:", RATIO, "| OFFSETS:", OFFSETS)
print("CONF_THRESH:", CONF_THRESH, "| CONF array length:", len(CONF))

Roll: 901503252 | seed: 279384723
FMAPS: [19, 5, 1] | BPC: [6, 6, 6] | NUM_CLASSES: 21
SCALE: 0.59 | RATIO: 0.5 | OFFSETS: [ 0.072 -0.029  0.196  0.028]
CONF_THRESH: 0.5 | CONF array length: 2322


## Q1 — Total default boxes (15 pts)
SSD predicts one set of boxes per cell of each feature map. For **your** `FMAPS` (each `f×f`) and `BPC` (boxes per cell), compute the number of boxes from each feature map and the total.

**Produce:** `per_map` (list) and `total_boxes` (int)

In [2]:
# per_map = boxes from each feature map (f*f*b); total_boxes = their sum.
# Produce: per_map, total_boxes
import numpy as np
per_map = [f * f * b for f, b in zip(FMAPS, BPC)]
total_boxes = int(sum(per_map))

print("per_map:", per_map)
print("total_boxes:", total_boxes)

per_map: [2166, 150, 6]
total_boxes: 2322


## Q2 — Prediction tensor shapes (15 pts)
Give the shapes of the **localization** and **confidence** outputs for a batch of 1, using `total_boxes` and `NUM_CLASSES`.

**Produce:** `loc_shape` and `conf_shape` (tuples)

In [3]:
# loc_shape = (1, total_boxes, 4)
# conf_shape = (1, total_boxes, NUM_CLASSES)
# Produce: loc_shape, conf_shape

loc_shape = (1, total_boxes, 4)
conf_shape = (1, total_boxes, NUM_CLASSES)

print("loc_shape:", loc_shape)
print("conf_shape:", conf_shape)

loc_shape: (1, 2322, 4)
conf_shape: (1, 2322, 21)


## Q3 — Default box shape from an aspect ratio (10 pts)
For scale `SCALE` and aspect ratio `RATIO`, a default box has width `SCALE·√RATIO` and height `SCALE/√RATIO`. Compute them.

**Produce:** `w, h`

In [4]:
# w = SCALE*sqrt(RATIO) ; h = SCALE/sqrt(RATIO). Produce: w, h
import numpy as np
w = float(SCALE * np.sqrt(RATIO))
h = float(SCALE / np.sqrt(RATIO))

print("w:", w)
print("h:", h)

w: 0.41719300090006306
h: 0.834386001800126


## Q4 — Decode predicted offsets (20 pts)
SSD predicts offsets to a default box. Decode `OFFSETS = (dcx, dcy, dw, dh)` against `DEFAULT_BOX = (cx, cy, w, h)` using:
`cx' = cx + dcx·w`, `cy' = cy + dcy·h`, `w' = w·exp(dw)`, `h' = h·exp(dh)`.

**Produce:** `pred_box` (array `[cx', cy', w', h']`)

In [5]:
# Decode OFFSETS against DEFAULT_BOX with the formulas above. Produce: pred_box
import numpy as np
cx, cy, default_w, default_h = DEFAULT_BOX
dcx, dcy, dw, dh = OFFSETS

cx_pred = cx + dcx * default_w
cy_pred = cy + dcy * default_h
w_pred = default_w * np.exp(dw)
h_pred = default_h * np.exp(dh)

pred_box = np.array([cx_pred, cy_pred, w_pred, h_pred])
print("pred_box:", pred_box)

pred_box: [0.5216     0.4942     0.36495807 0.20567914]


## Q5 — Multi-scale detection (10 pts)
Among **your** `FMAPS`, identify which feature map (by size) is best for detecting **small** objects and which for **large** objects.

**Produce:** `small_obj_map, large_obj_map`

In [6]:
# small objects -> highest-resolution (largest f) ; large objects -> smallest f.
# Produce: small_obj_map, large_obj_map
small_obj_map = int(max(FMAPS))
large_obj_map = int(min(FMAPS))

print("small_obj_map:", small_obj_map)
print("large_obj_map:", large_obj_map)

small_obj_map: 19
large_obj_map: 1


## Q6 — Confidence thresholding (15 pts)
SSD predicts thousands of boxes, but most are low-confidence background that gets discarded before NMS. Using your per-box confidence array `CONF` and threshold `CONF_THRESH`, count how many boxes survive the threshold, and what **percentage** of all boxes that is.

**Produce:** `n_kept` (int) and `pct_kept` (float, %)

In [7]:
# Keep boxes with CONF >= CONF_THRESH. Produce: n_kept, pct_kept
import numpy as np
n_kept = int(np.sum(CONF >= CONF_THRESH))
pct_kept = float((n_kept / len(CONF)) * 100.0)

print("n_kept:", n_kept)
print(f"pct_kept: {pct_kept:.2f}%")

n_kept: 1154
pct_kept: 49.70%


## Q7 — Short answer, no code (15 pts)
Answer in your own words:

**(a)** The canonical SSD300 produces **8732** default boxes. Why does predicting so many boxes help, given that most are discarded?

> *(a) Predicting a vast number of candidate boxes of varying scales and aspect ratios ensures extensive spatial coverage of the image. This guarantees that any real object will overlap heavily with at least one default box, allowing stable training and a very high recall rate during inference, even if background predictions are ultimately filtered out.*

**(b)** Why does each box need exactly **4** localization values but `NUM_CLASSES` confidence values?

> *(b) A bounding box is uniquely defined by 4 spatial parameters (cx, cy, w, h), which requires 4 localization values. However, for classification, the model needs to output a complete probability distribution over all possible categories (including background) for each box, requiring NUM_CLASSES values.*


**(c)** Why does SSD predict **offsets** to a default box rather than the absolute box coordinates directly?

> *(c) Predicting relative offsets simplifies the optimization surface significantly. Offsets are scale-invariant and bounded, making the learning objective stable and fast to converge compared to trying to regress absolute coordinates over unpredictable scales.*

**(d)** SSD uses **several feature maps of different resolutions**. Why is a higher-resolution feature map better for small objects?

> *(d) High-resolution feature maps preserve fine-grained details and keep spatial coordinates highly localized because they have undergone fewer downsampling layers. Each grid cell on a larger feature map has a smaller receptive field, making it well-suited to identify and localize tiny objects.*


## Submission Summary
Run last; keep the output.

In [8]:
# === SUBMISSION SUMMARY ===
print("Roll:", ROLL_NUMBER, "| seed:", seed)
print("Q1 total boxes:", total_boxes, "| FMAPS", FMAPS, "BPC", BPC)
print("Q2 loc/conf:", loc_shape, conf_shape)
print("Q3 default w,h:", round(w,4), round(h,4))
print("Q4 decoded box:", np.round(pred_box,4))
print("Q5 small/large fmap:", small_obj_map, large_obj_map)
print("Q6 kept:", n_kept, "of", len(CONF), f"({round(pct_kept,1)}%)", "| CONF_THRESH", CONF_THRESH)

Roll: 901503252 | seed: 279384723
Q1 total boxes: 2322 | FMAPS [19, 5, 1] BPC [6, 6, 6]
Q2 loc/conf: (1, 2322, 4) (1, 2322, 21)
Q3 default w,h: 0.4172 0.8344
Q4 decoded box: [0.5216 0.4942 0.365  0.2057]
Q5 small/large fmap: 19 1
Q6 kept: 1154 of 2322 (49.7%) | CONF_THRESH 0.5
